# Do plans with more benefits have higher rates?

In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, ShortType
)

# ── Paths ──────────────────────────────────────────────────────────────────

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
ALL_DATA_DIR = os.path.join(BASE_DIR, "data", "all_data_hhs")
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")
os.makedirs(ALL_DATA_DIR, exist_ok=True)
os.makedirs(BRONZE_PARQUET_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"ALL_DATA_DIR: {ALL_DATA_DIR}")
print(f"BRONZE_PARQUET_DIR: {BRONZE_PARQUET_DIR}")

In [0]:
# SCHEMA Defintions for BenefitsCost Sharing and Rates csv's

BENEFITS_SCHEMA = StructType([
    StructField("BenefitName",       StringType(), True),
    StructField("BusinessYear",      ShortType(),  True),
    StructField("CoinsInnTier1",     StringType(), True),
    StructField("CoinsInnTier2",     StringType(), True),
    StructField("CoinsOutofNet",     StringType(), True),
    StructField("CopayInnTier1",     StringType(), True),
    StructField("CopayInnTier2",     StringType(), True),
    StructField("CopayOutofNet",     StringType(), True),
    StructField("EHBVarReason",      StringType(), True),
    StructField("Exclusions",        StringType(), True),
    StructField("Explanation",       StringType(), True),
    StructField("ImportDate",        StringType(), True),
    StructField("IsCovered",         StringType(), True),
    StructField("IsEHB",             StringType(), True),
    StructField("IsExclFromInnMOOP", StringType(), True),
    StructField("IsExclFromOonMOOP", StringType(), True),
    StructField("IsStateMandate",    StringType(), True),
    StructField("IsSubjToDedTier1",  StringType(), True),
    StructField("IsSubjToDedTier2",  StringType(), True),
    StructField("IssuerId",          StringType(), True),
    StructField("IssuerId2",         StringType(), True),
    StructField("LimitQty",          StringType(), True),
    StructField("LimitUnit",         StringType(), True),
    StructField("MinimumStay",       StringType(), True),
    StructField("PlanId",            StringType(), True),
    StructField("QuantLimitOnSvc",   StringType(), True),
    StructField("RowNumber",         IntegerType(), True),
    StructField("SourceName",        StringType(), True),
    StructField("StandardComponentId",StringType(),True),
    StructField("StateCode",         StringType(), True),
    StructField("StateCode2",        StringType(), True),
    StructField("VersionNum",        IntegerType(), True),
])

# ── Rate ───────────────────────────────────────────────────────────────────
RATE_SCHEMA = StructType([
    StructField("BusinessYear",                              ShortType(),  True),
    StructField("StateCode",                                StringType(), True),
    StructField("IssuerId",                                 StringType(), True),
    StructField("SourceName",                               StringType(), True),
    StructField("VersionNum",                               IntegerType(), True),
    StructField("ImportDate",                               StringType(), True),
    StructField("IssuerId2",                                StringType(), True),
    StructField("FederalTIN",                               StringType(), True),
    StructField("RateEffectiveDate",                        StringType(), True),
    StructField("RateExpirationDate",                       StringType(), True),
    StructField("PlanId",                                   StringType(), True),
    StructField("RatingAreaId",                             StringType(), True),
    StructField("Tobacco",                                  StringType(), True),
    StructField("Age",                                      StringType(), True),
    StructField("IndividualRate",                           StringType(), True),
    StructField("IndividualTobaccoRate",                    StringType(), True),
    StructField("Couple",                                   StringType(), True),
    StructField("PrimarySubscriberAndOneDependent",         StringType(), True),
    StructField("PrimarySubscriberAndTwoDependents",        StringType(), True),
    StructField("PrimarySubscriberAndThreeOrMoreDependents",StringType(), True),
    StructField("CoupleAndOneDependent",                   StringType(), True),
    StructField("CoupleAndTwoDependents",                  StringType(), True),
    StructField("CoupleAndThreeOrMoreDependents",          StringType(), True),
    StructField("RowNumber",                                IntegerType(), True),
])

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType

def read_data_from_csv(filepath: str, schema: StructType) -> DataFrame:
    """Read a CSV with an explicit schema."""
    reader = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("delimiter", ",")
        .option("escape", '"')
        .option("multiLine", True)
        .schema(schema)
    )
    df = reader.load(filepath)
    return df

In [0]:
# Write data to parquet
def write(input_df: DataFrame, out_dir):
    return input_df.write.mode('overwrite').parquet(out_dir)

In [0]:
# Write data to parquet for each file. TODO: Mask the ITIN here.
bronze = {
    "benefits":     read_data_from_csv(f"{BASE_DIR}/BenefitsCostSharing.csv",  BENEFITS_SCHEMA),
    "rates":        read_data_from_csv(f"{BASE_DIR}/Rate.csv",  RATE_SCHEMA)
}

# Mask FederalTIN in rates
bronze["rates"] = bronze["rates"].withColumn(
    "FederalTIN",
    F.when(F.col("FederalTIN").isNotNull(), F.sha2(F.col("FederalTIN"), 256)).otherwise(None)
)


print("Bronze DataFrames loaded:")
for name, df in bronze.items():
    write(df, f"{BRONZE_PARQUET_DIR}/{name}")
    print(f"  {name}  {df.count()}  ...")
 